In [ ]:
import sys
import json
import h5py
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.voltage.extraction import  load_voltage_roi_transform_h5

display(HTML("<style>.container { width:100% !important; }</style>"))
warnings.filterwarnings("default")

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib notebook

In [ ]:
# ---------------------------------------------------------------------
# Local repository / data paths
# ---------------------------------------------------------------------
# If running from inside the repo, this can usually stay as None.
# If imports fail, set REPO_ROOT to your local clone, e.g.
# REPO_ROOT = Path(r"C:\Users\andrew.shelton\Dropbox\allen institute\Python_Code\ams\ophys\vip-slap2-analysis")
REPO_ROOT = None

if REPO_ROOT is not None:
    src_path = Path(REPO_ROOT) / "src"
    if str(src_path) not in sys.path:
        sys.path.insert(0, str(src_path))

BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")

# Optional local copy of the summary table. The registry discovers sessions from BASE_PATH;
# this file is only useful for ad hoc inspection or manual cross-checks.
SUMMARY_XLSX = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\VIP_SD_summary.xlsx")

# ---------------------------------------------------------------------
# Session selection
# ---------------------------------------------------------------------
target_mice = [
    826031,
    826032,
]

EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]
PARADIGMS = ["change_detection_passive"]

In [ ]:
registry = VIPSessionRegistry.from_basepath(BASE_PATH)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
    paradigms=PARADIGMS,
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

print(f"Found {len(assets)} candidate sessions for target mice: {target_mice}")
display(process_df.head())

In [ ]:
session_dir = assets[0].derived_dir / 'voltage' / 'voltage_session_traces_dff_robust_f0_trial.h5'

In [ ]:

asset = assets[0]
dmd = 1
roi_index = 0
trace_signal = "dff_robust_f0"

session_trace_h5 = asset.derived_dir / 'voltage' / 'voltage_session_traces_dff_robust_f0_trial.h5'
print("Inspecting:", session_trace_h5)

if not session_trace_h5.exists():
    print("Session trace H5 not found. Run voltage extraction first, or use the next cell to compute one ROI on the fly.")
else:
    roi = load_voltage_roi_transform_h5(session_trace_h5, dmd=dmd, roi_index=roi_index)
    t = roi["timebase_sec"]
    raw_f = roi["raw_f"]
    f0 = roi["f0"]
    dff = (raw_f-f0)/f0
#         dff = roi["dff"]

    fig, ax = plt.subplots(figsize=(15, 4))
    ax.plot(t, raw_f, lw=0.4, label="raw F")
    ax.plot(t, f0, lw=1.0, label="F0")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Fluorescence")
    ax.set_title(f"{asset.session_id} | DMD{dmd} ROI {roi_index} | raw F and F0")
    ax.legend()
    plt.show()

    fig, ax = plt.subplots(figsize=(15, 4))
    ax.plot(t, dff, lw=0.4)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("dFF = (F0 - F) / F0")
    ax.set_title(f"{asset.session_id} | DMD{dmd} ROI {roi_index} | ASAP7y dFF")
    plt.show()

In [ ]:
image_times = 